# Langchain: The basics

Please install the following libraries in your virtual environment. I have used the following specified versions.
If you are using more recent versions (say: langchain version 1.x) lot of code might not work and you may need to change it a bit. 

- **getpass** (`pip install getpass4`)
- **langchain** (`pip install langchain==0.3.27`)
- **langchain-groq** (`pip install langchain-groq==0.2.0`)

In [1]:
import os
from getpass import getpass  # to enter the password securely

In [2]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

## Zero shot prompting

In [3]:
from langchain_groq import ChatGroq

In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
    max_tokens=256,
    max_retries=3,
)

In [5]:
prompt = """What is the sentiment of the customer review given below? It should be either Positive, Negative or Neutral.
Customer Review: The product quality is excellent and delivery was prompt.
"""

In [6]:
print(llm.invoke(prompt))

content='The sentiment of the customer review is: Positive. The customer uses words like "excellent" and "prompt" to describe their experience, indicating a very favorable opinion.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 69, 'total_tokens': 104, 'completion_time': 0.099635867, 'prompt_time': 0.004875565, 'queue_time': 0.057601475, 'total_time': 0.104511432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3f3b593e33', 'finish_reason': 'stop', 'logprobs': None} id='run--79b37192-8f2c-4369-8336-e276025b570d-0' usage_metadata={'input_tokens': 69, 'output_tokens': 35, 'total_tokens': 104}


## Prompt Templates

In [7]:
from langchain import PromptTemplate

In [8]:
restaurant_template = """
What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: {review_text}
Sentiment:
"""

review_prompt = PromptTemplate(
    input_variables=["review_text"],
    template=restaurant_template,
)

In [9]:
reviews = [
    "The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.",
    "Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.",
    "The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!",
    "The food tasted alright, but the tables were not very clean, which was off-putting.",
    "Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit."
]

In [10]:
print(review_prompt.format(review_text=reviews[4]))


What is the sentiment of the customer review given below? It should be a positive or negative sentiment.

review: Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit.
Sentiment:



In [11]:
review_chain = review_prompt | llm

In [12]:
for review in reviews:
    print(f"Review: {review}")
    print(review_chain.invoke({"review_text": review}).content)
    print("-" * 50)

Review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
Sentiment: Positive
--------------------------------------------------
Review: Impressive service! The staff was attentive and made excellent recommendations. Thoroughly enjoyed the evening.
Sentiment: Positive
--------------------------------------------------
Review: The restaurant's interior was a visual treat, beautifully paired with their gourmet dishes. A delightful experience!
Sentiment: Positive
--------------------------------------------------
Review: The food tasted alright, but the tables were not very clean, which was off-putting.
Sentiment: Negative

Although the customer mentions that "The food tasted alright", which is a neutral or slightly positive comment, the negative aspect of the experience ("the tables were not very clean, which was off-putting") seems to outweigh the positive, resulting in an overall negative sentiment.
----------------------------------------

## Few shot prompting

In [13]:
from langchain import FewShotPromptTemplate

In [14]:
# First, create the list of few shot examples.
examples = [
    {
        "Review": "The grilled chicken was seasoned to perfection and simply melted in the mouth.",
        "Category": "Food Quality"
    },
    {
        "Review": "Despite the crowd, the place was immaculately clean and the restrooms were spotless.",
        "Category": "Overall Hygiene"
    },
    {
        "Review": "The dim lighting and soothing jazz music provided an intimate and romantic setting.",
        "Category": "Restaurant Ambience"
    },
    {
        "Review": "We were kept waiting for our table even after a confirmed reservation and the staff seemed disinterested.",
        "Category": "Customer Service"
    }
]

In [16]:
example_formatter_template = """
review: {Review}
category: {Category}\n
"""
category_prompt = PromptTemplate(
    input_variables=["Review", "Category"],
    template=example_formatter_template,
)

print(category_prompt.format(**examples[0]))


review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
category: Food Quality




In [17]:
# Finally, we create the `FewShotPromptTemplate` object.

few_shot_category_prompt = FewShotPromptTemplate(
    # These are the examples we want to insert into the prompt.
    examples=examples,
    # This is how we want to format the examples when we insert them into the prompt.
    example_prompt=category_prompt,
    # The prefix is some text that goes before the examples in the prompt.
    # Usually, this consists of intructions.
    prefix="Classify the reviews into one the four categories as given in the examples.",
    # The suffix is some text that goes after the examples in the prompt.
    # Usually, this is where the user input will go
    suffix="Review: {review_text}\nCategory:",
    # The input variables are the variables that the overall prompt expects.
    input_variables=["review_text"],
    # The example_separator is the string we will use to join the prefix, examples, and suffix together with.
    example_separator="\n",
)

In [18]:
# We can now generate a prompt using the `format` method.
print(few_shot_category_prompt.format(review_text=reviews[0]))

Classify the reviews into one the four categories as given in the examples.

review: The grilled chicken was seasoned to perfection and simply melted in the mouth.
category: Food Quality



review: Despite the crowd, the place was immaculately clean and the restrooms were spotless.
category: Overall Hygiene



review: The dim lighting and soothing jazz music provided an intimate and romantic setting.
category: Restaurant Ambience



review: We were kept waiting for our table even after a confirmed reservation and the staff seemed disinterested.
category: Customer Service


Review: The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
Category:


In [19]:
category_chain = few_shot_category_prompt | llm

print(reviews[0])
# Run the chain only specifying the input variable.
print(category_chain.invoke(reviews[0]))


The seafood platter was fresh and flavorful. Loved the chic décor and the ambiance of the place.
content='Since the review mentions both the food ("The seafood platter was fresh and flavorful") and the ambiance ("Loved the chic décor and the ambiance of the place"), it can be classified under two categories. However, based on the given options, the review can be categorized as either "Food Quality" or "Restaurant Ambience". \n\nGiven the equal emphasis on both aspects, it\'s difficult to choose one. But if I had to pick one, I\'d categorize it as: \n\nCategory: Food Quality \n\n(Note: A more accurate classification might be a combination of "Food Quality" and "Restaurant Ambience", but that option is not provided.)' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 170, 'total_tokens': 302, 'completion_time': 0.438029766, 'prompt_time': 0.010692651, 'queue_time': 0.058731599, 'total_time': 0.448722417}, 'model_name': 'llama-3.3-70b-versa

## Chaining Multiple Prompts

In [21]:
from langchain.schema import StrOutputParser

In [22]:
review_chain = review_prompt | llm | StrOutputParser()

In [23]:
review_chain.invoke({"review_text": reviews[0]})

'Sentiment: Positive'

In [24]:
category_chain = few_shot_category_prompt | llm | StrOutputParser()

In [25]:
category_chain.invoke({"review_text": reviews[1]})

'Category: Customer Service'

In [26]:
response_to_customer = """
"Write an appropriate response to the customer by either appreciating their positive experience or adressing the specific concern that customer might
have mentioned in the feedback below.

<feedback>
{feedback}
</feedback>

The feedback is about {category}.

The goal of the response is to ensure customer engagement.
"""

In [27]:
response_prompt = PromptTemplate(
    input_variables=["feedback", "category"],
    template=response_to_customer,
)

In [29]:
final_response_chain = response_prompt | llm | StrOutputParser()

In [30]:
complete_chain = (
    {
        "feedback": review_chain,
        "category": category_chain,
    }
    | final_response_chain
)

In [31]:
from pprint import pprint

In [33]:
reviews[4]

'Waited 30 minutes even with a reservation, and the main course was served cold. Disappointing visit.'

In [34]:
review_chain.invoke({"review_text": reviews[4]})

'Negative. \n\nThe customer mentions waiting for 30 minutes despite having a reservation, and their main course was served cold, which led to a "disappointing visit". These details all contribute to an overall negative sentiment.'

In [35]:
category_chain.invoke({"review_text": reviews[4]})

'Based on the given examples, I would categorize the review as:\n\nCategory: Customer Service\n\nReason: The review mentions waiting for a table despite a confirmed reservation, which relates to the service provided by the staff. Additionally, the main course being served cold is also a service issue, as it indicates a lapse in the quality of service provided.'

In [32]:
pprint(complete_chain.invoke({"review_text": reviews[4]}))

('Dear valued customer,\n'
 '\n'
 'I am truly sorry to hear that your recent experience at our establishment '
 'did not meet your expectations. We apologize for the inconvenience you faced '
 'while waiting for a table despite having a reservation, and we are '
 'especially disappointed to learn that your main course was served cold.\n'
 '\n'
 'At our restaurant, we take pride in providing excellent customer service and '
 'ensuring that every guest has a memorable dining experience. Clearly, we '
 'fell short of this standard in your case, and for that, we are truly sorry.\n'
 '\n'
 'We would like to make things right and invite you to give us another chance '
 'to serve you better. Could you please contact us directly so we can discuss '
 'your experience in more detail and offer a suitable resolution? Your '
 'feedback is invaluable in helping us to identify areas for improvement, and '
 'we appreciate the time you took to share your concerns with us.\n'
 '\n'
 'Thank you for your 